# Flight Delay Prediction  Phase 1
## Data Preparation, Feature Engineering & Exploratory Data Analysis

This notebook loads the cleaned flight + weather dataset (`cleaned_final_train.csv`) shipped with the repository, engineers features, explores the data, and trains baseline classifiers.

> The original notebooks parsed raw `.docx`/`.xlsx` source files that are not distributed with this project. They have been rewired to run end-to-end from the cleaned CSV instead.

## 1. Load the data

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd

CSV_PATH = "cleaned_final_train.csv"   # the cleaned dataset shipped with this repo

def load_and_prepare(path=CSV_PATH):
    """Load the cleaned flight+weather dataset and fix known quirks."""
    df = pd.read_csv(path)

    # The weather column headers were saved with a mis-encoded degree sign, e.g.
    # "Temperature (�F)_Avg". Strip any non-ASCII bytes so the names become
    # clean "(F)" labels, then collapse any doubled whitespace.
    df.columns = [re.sub(r"\s+", " ", re.sub(r"[^\x00-\x7F]", "", c)).strip()
                  for c in df.columns]

    # Parse the scheduled departure timestamp into a real datetime.
    df["Scheduled Time"] = pd.to_datetime(df["Departure_Scheduled"], errors="coerce")

    # Friendly name for the prediction target (minutes of departure delay).
    df = df.rename(columns={"Departure_Delay_Minutes": "Delay (minutes)"})

    # Drop the few rows without a usable timestamp or target.
    df = df.dropna(subset=["Scheduled Time", "Delay (minutes)"]).reset_index(drop=True)
    return df

df = load_and_prepare()
print("Loaded:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Feature engineering

In [ ]:

# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------
def assign_season(month):
    return {12: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1,
            6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3}[month]

def add_features(df):
    df = df.copy()

    # Composite "weather severity" index (wind + dryness + temperature swing).
    df["Weather Severity"] = (
        df["Wind Speed (mph)_Max"] * 0.4
        + (100 - df["Humidity (%)_Avg"]) * 0.3
        + (df["Temperature (F)_Max"] - df["Temperature (F)_Min"]) * 0.3
    )

    # Temporal features from the scheduled timestamp.
    hour = df["Scheduled Time"].dt.hour
    df["IsWeekend"] = (df["Scheduled Time"].dt.weekday >= 5).astype(int)
    df["PeakHour"] = (hour.between(6, 9) | hour.between(17, 20)).astype(int)
    df["Season"] = df["Scheduled Time"].dt.month.map(assign_season)
    df["Scheduled Hour"] = hour
    df["Scheduled Weekday"] = df["Scheduled Time"].dt.weekday
    df["Scheduled Month"] = df["Scheduled Time"].dt.month

    # Airport-level average delay (departure airport reputation).
    df["Avg Departure Delay (Airport)"] = (
        df.groupby("Departure_IATA")["Delay (minutes)"].transform("mean")
    )
    return df

df = add_features(df)

# Columns used by every model below.
NUMERIC_FEATURES = [
    "Temperature (F)_Max", "Temperature (F)_Avg", "Temperature (F)_Min",
    "Dew Point (F)_Max", "Dew Point (F)_Avg", "Dew Point (F)_Min",
    "Humidity (%)_Max", "Humidity (%)_Avg", "Humidity (%)_Min",
    "Wind Speed (mph)_Max", "Wind Speed (mph)_Avg", "Wind Speed (mph)_Min",
    "Pressure (in)_Max", "Pressure (in)_Avg", "Pressure (in)_Min",
    "Hour_of_Day", "Week_Number", "Scheduled Hour", "Scheduled Weekday",
    "Scheduled Month", "Weather Severity", "IsWeekend", "PeakHour", "Season",
    "Avg Departure Delay (Airport)",
]
CATEGORICAL_FEATURES = [
    "Departure_IATA", "Arrival_IATA", "Airline_IATA", "Day_of_Week", "Month_of_Year",
]

# Impute any residual missing values so the models never see NaNs.
for c in NUMERIC_FEATURES:
    df[c] = df[c].fillna(df[c].median())
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].fillna("unknown")

print("Numeric features :", len(NUMERIC_FEATURES))
print("Categorical      :", len(CATEGORICAL_FEATURES))
df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].head()

## 3. Exploratory Data Analysis

### 3.1 Distribution of departure delays

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="Delay (minutes)", bins=50, kde=True, color="steelblue")
plt.title("Distribution of Flight Departure Delays")
plt.xlabel("Delay (minutes)")
plt.ylabel("Frequency")
plt.show()

print(df["Delay (minutes)"].describe())
print("On-time (0 min) share: {:.1%}".format((df["Delay (minutes)"] == 0).mean()))

### 3.2 Average delay by hour of day and by weekday

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=df, x="Scheduled Hour", y="Delay (minutes)",
            errorbar=None, palette="plasma", ax=axes[0])
axes[0].set_title("Average Delay by Scheduled Hour")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Average Delay (minutes)")

weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_day = df.groupby("Scheduled Weekday")["Delay (minutes)"].mean()
sns.lineplot(x=by_day.index, y=by_day.values, marker="o", ax=axes[1])
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(weekday_names)
axes[1].set_title("Average Delay by Weekday")
axes[1].set_xlabel("Weekday")
axes[1].set_ylabel("Average Delay (minutes)")
plt.tight_layout()
plt.show()

### 3.3 Delay by departure airport and airline

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

dep = df.groupby("Departure_IATA")["Delay (minutes)"].mean().sort_values()
dep.plot(kind="bar", color="skyblue", ax=axes[0])
axes[0].set_title("Average Delay by Departure Airport")
axes[0].set_ylabel("Average Delay (minutes)")

top_airlines = df["Airline_IATA"].value_counts().head(10).index
air = (df[df["Airline_IATA"].isin(top_airlines)]
       .groupby("Airline_IATA")["Delay (minutes)"].mean().sort_values())
air.plot(kind="bar", color="salmon", ax=axes[1])
axes[1].set_title("Average Delay  Top 10 Airlines by Volume")
axes[1].set_ylabel("Average Delay (minutes)")
plt.tight_layout()
plt.show()

### 3.4 Correlation between weather features and delay

In [ ]:

weather_cols = [
    "Temperature (F)_Avg", "Dew Point (F)_Avg", "Humidity (%)_Avg",
    "Wind Speed (mph)_Avg", "Pressure (in)_Avg", "Weather Severity",
    "Delay (minutes)",
]
plt.figure(figsize=(9, 7))
sns.heatmap(df[weather_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation: Weather Features vs Delay")
plt.show()

## 4. Baseline classification models

We build a reusable preprocessing pipeline, then train baseline binary and multi-class classifiers.

In [ ]:

# ---------------------------------------------------------------------------
# Reusable preprocessing pipeline: scale numbers + one-hot encode categories
# ---------------------------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
print("Design matrix:", X.shape)

### 4.1 Binary classification  On-time vs Delayed

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)

y_binary = (df["Delay (minutes)"] > 0).astype(int)   # 0 = on-time, 1 = delayed
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, stratify=y_binary, random_state=42)

binary_clf = Pipeline([("pre", make_preprocessor()),
                       ("clf", RandomForestClassifier(n_estimators=100, max_depth=20,
                                                      random_state=42, n_jobs=-1))])
binary_clf.fit(X_train, y_train)
y_pred = binary_clf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=["On-Time", "Delayed"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["On-Time", "Delayed"]).plot(cmap="Blues")
plt.title("Binary Classification  Confusion Matrix")
plt.show()

### 4.2 Multi-class classification  delay severity buckets

Buckets: **No Delay** (0 min), **Short** (<45), **Moderate** (45175), **Long** (>175). Only the buckets present in the data are modelled.

In [ ]:

delay_bucket = pd.cut(
    df["Delay (minutes)"], bins=[-1, 0, 45, 175, float("inf")],
    labels=["No Delay", "Short Delay", "Moderate Delay", "Long Delay"],
).astype(str)
print("Class counts:\n", delay_bucket.value_counts(), "\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, delay_bucket, test_size=0.2, stratify=delay_bucket, random_state=42)

multi_clf = Pipeline([("pre", make_preprocessor()),
                      ("clf", RandomForestClassifier(n_estimators=100, max_depth=20,
                                                     random_state=42, n_jobs=-1))])
multi_clf.fit(X_train, y_train)
y_pred = multi_clf.predict(X_test)

labels = sorted(delay_bucket.unique())
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues", xticks_rotation=45)
plt.title("Multi-Class Classification  Confusion Matrix")
plt.show()

### 4.3 Quick model comparison (binary task)

Fast, interpretable models compared on the on-time-vs-delayed task.

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

y_binary = (df["Delay (minutes)"] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, stratify=y_binary, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
}

scores = {}
for name, model in models.items():
    pipe = Pipeline([("pre", make_preprocessor()), ("clf", model)])
    pipe.fit(X_train, y_train)
    scores[name] = accuracy_score(y_test, pipe.predict(X_test))
    print(f"{name:<22} accuracy = {scores[name]:.4f}")

plt.figure(figsize=(8, 5))
sns.barplot(x=list(scores.keys()), y=list(scores.values()), palette="husl")
plt.title("Model Accuracy Comparison (Binary Task)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

---
**Phase 1 complete.** Continue to **Phase 2** for hyper-parameter tuning, regression, and Kaggle-style submission files.